# MLOps para Classificação LULC com SITS + Brazil Data Cube

Pipeline: TerraClass 2024 (Maranhão, região de Balsas) → cubo Sentinel-2 do BDC → séries temporais → treino com SITS → tracking no MLflow.

**Infra esperada rodando:**
- `docker compose up -d` (MLflow em localhost:5000)
- kernel do ambiente mamba `mlops_lulc`

## 0. Setup e configuração

In [1]:
from pathlib import Path

import geopandas as gpd
import mlflow
import numpy as np
import pandas as pd
from shapely.geometry import box

DATA_DIR = Path("../data")
SHP_PATH = DATA_DIR / "CER.2024.MARANHAO.21.V" / "CER.2024.MARANHAO.21.V.shp"
OUTPUT_DIR = Path("../data/processed")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MLFLOW_TRACKING_URI = "http://localhost:5000"
EXPERIMENT_NAME = "sits-terraclass-balsas"

# bbox pequeno em torno de Balsas-MA (~35km x 35km)
ROI = {
    "lon_min": -46.20,
    "lat_min": -7.70,
    "lon_max": -45.85,
    "lat_max": -7.35,
}

START_DATE = "2024-01-01"
END_DATE = "2024-01-30"  # período reduzido para economizar memória; ampliar depois

N_POINTS_PER_CLASS = 10
SEED = 42

mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='/mlflow/artifacts/1', creation_time=1788636625097, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1788636625097, lifecycle_stage='active', name='sits-terraclass-balsas', tags={}, trace_location=None, workspace='default'>

## 1. Carregar TerraClass 2024 (vetorial)

Shapefile do TerraClass em EPSG:4674 (SIRGAS 2000). Cada linha é um multipolígono por classe.

In [2]:
gdf = gpd.read_file(SHP_PATH)
print(f"CRS: {gdf.crs}")
print(f"Colunas: {list(gdf.columns)}")
print(f"Classes ({len(gdf)}):")
print(gdf["CLASSE"].value_counts())
print(f"bbox total: {gdf.total_bounds}")

CRS: EPSG:4674
Colunas: ['VALUE', 'CLASSE', 'Area_Km', 'geometry']
Classes (14):
CLASSE
VEGETACAO_NATURAL_PRIMARIA                        1
VEGETACAO_NATURAL_SECUNDARIA                      1
SILVICULTURA                                      1
PASTAGEM                                          1
CULTURA_AGRICOLA_PERENE                           1
CULTURA_AGRICOLA_SEMIPERENE                       1
CULTURA_AGRICOLA_TEMPORARIA_DE_1_CICLO            1
CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO    1
MINERACAO                                         1
URBANIZADA                                        1
OUTROS_USOS                                       1
OUTRAS_AREAS_EDIFICADAS                           1
DESFLORESTAMENTO_NO_ANO                           1
CORPO_DAGUA                                       1
Name: count, dtype: int64
bbox total: [-47.90064216 -10.26171917 -41.79585216  -2.33208917]


## 2. Recortar para a área de estudo (Balsas-MA)

A cena cobre grande parte do Maranhão; recortamos para um bbox pequeno na fronteira agrícola.

In [3]:
bbox = box(ROI["lon_min"], ROI["lat_min"], ROI["lon_max"], ROI["lat_max"])
gdf_clip = gpd.clip(gdf, bbox).to_crs("EPSG:4326")

print(f"Classes no recorte ({len(gdf_clip)}):")
print(gdf_clip["CLASSE"].value_counts())
print(f"bbox recorte: {gdf_clip.total_bounds}")

Classes no recorte (12):
CLASSE
CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO    1
CULTURA_AGRICOLA_TEMPORARIA_DE_1_CICLO            1
PASTAGEM                                          1
MINERACAO                                         1
URBANIZADA                                        1
OUTROS_USOS                                       1
VEGETACAO_NATURAL_SECUNDARIA                      1
OUTRAS_AREAS_EDIFICADAS                           1
VEGETACAO_NATURAL_PRIMARIA                        1
CORPO_DAGUA                                       1
DESFLORESTAMENTO_NO_ANO                           1
SILVICULTURA                                      1
Name: count, dtype: int64
bbox recorte: [-46.2   -7.7  -45.85  -7.35]


## 3. Gerar pontos de amostra a partir dos polígonos

O SITS espera amostras pontuais (`longitude`, `latitude`, `label`, `start_date`, `end_date`). Amostramos N pontos aleatórios por classe dentro dos polígonos.

In [4]:
rng = np.random.default_rng(SEED)
sample_rows = []

for _, row in gdf_clip.iterrows():
    pts_series = gpd.GeoSeries([row.geometry], crs="EPSG:4326").sample_points(
        N_POINTS_PER_CLASS, rng=rng
    )
    for pt in pts_series.iloc[0].geoms:
        sample_rows.append(
            {
                "longitude": pt.x,
                "latitude": pt.y,
                "label": row["CLASSE"],
                "start_date": START_DATE,
                "end_date": END_DATE,
            }
        )

samples_df = pd.DataFrame(sample_rows)
print(samples_df.shape)
print(samples_df["label"].value_counts())
samples_df.head()

(120, 5)
label
CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO    10
CULTURA_AGRICOLA_TEMPORARIA_DE_1_CICLO            10
PASTAGEM                                          10
MINERACAO                                         10
URBANIZADA                                        10
OUTROS_USOS                                       10
VEGETACAO_NATURAL_SECUNDARIA                      10
OUTRAS_AREAS_EDIFICADAS                           10
VEGETACAO_NATURAL_PRIMARIA                        10
CORPO_DAGUA                                       10
DESFLORESTAMENTO_NO_ANO                           10
SILVICULTURA                                      10
Name: count, dtype: int64


,longitude,latitude,label,start_date,end_date
0,-46.149417,-7.531194,CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO,2024-01-01,2024-01-30
1,-46.184784,-7.408290,CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO,2024-01-01,2024-01-30
2,-45.969748,-7.644836,CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO,2024-01-01,2024-01-30
3,-45.966059,-7.624895,CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO,2024-01-01,2024-01-30
4,-45.858532,-7.620466,CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO,2024-01-01,2024-01-30


In [ ]:
samples_df.to_csv(OUTPUT_DIR / "samples.csv", index=False)

## 4. Criar cubo Sentinel-2 do BDC via pysits

⚠️ Célula pesada: carrega o R via rpy2.

In [5]:
from pysits import sits_cube, sits_timeline

cube = sits_cube(
    source="BDC",
    collection="SENTINEL-2-16D",
    bands=["NDVI"],
    start_date=START_DATE,
    end_date=END_DATE,
    roi=ROI,
    multicores=1,
)
print(cube)
print(sits_timeline(cube))

Error importing in API mode: ModuleNotFoundError("No module named '_rinterface_cffi_api'")
Trying to import in ABI mode.


  |======================================================================| 100%
  source      collection   satellite sensor    tile       xmin       xmax  \
0    BDC  SENTINEL-2-16D  SENTINEL-2    MSI  030013  5792000.0  5897600.0   
1    BDC  SENTINEL-2-16D  SENTINEL-2    MSI  030014  5792000.0  5897600.0   

         ymin        ymax                                                crs  \
0  10475200.0  10580800.0  PROJCRS["unknown",\n    BASEGEOGCRS["unknown",...   
1  10369600.0  10475200.0  PROJCRS["unknown",\n    BASEGEOGCRS["unknown",...   

                                           file_info  
0                           fid        date  band...  
1                           fid        date  band...  
['2024-01-01', '2024-01-17']


## 5. Extrair séries temporais dos pontos de amostra

In [6]:
from pysits import sits_get_data

time_series = sits_get_data(
    cube=cube,
    samples=samples_df,
    multicores=1,
)
print(time_series)

     longitude  latitude  start_date    end_date  \
0   -46.191299 -7.536416  2024-01-01  2024-01-17   
1   -46.184784 -7.408290  2024-01-01  2024-01-17   
2   -46.176944 -7.453504  2024-01-01  2024-01-17   
3   -46.173460 -7.696169  2024-01-01  2024-01-17   
4   -46.173450 -7.591787  2024-01-01  2024-01-17   
..         ...       ...         ...         ...   
115 -45.861946 -7.367745  2024-01-01  2024-01-17   
116 -45.860248 -7.546433  2024-01-01  2024-01-17   
117 -45.858532 -7.620466  2024-01-01  2024-01-17   
118 -45.855574 -7.483762  2024-01-01  2024-01-17   
119 -45.852761 -7.485858  2024-01-01  2024-01-17   

                                              label            cube  \
0    CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO  SENTINEL-2-16D   
1    CULTURA_AGRICOLA_TEMPORARIA_DE_MAIS_DE_1_CICLO  SENTINEL-2-16D   
2                        VEGETACAO_NATURAL_PRIMARIA  SENTINEL-2-16D   
3                                       CORPO_DAGUA  SENTINEL-2-16D   
4                   

## 6. Validação cruzada + treino com tracking no MLflow

In [7]:
from pysits import sits_kfold_validate, sits_rfor, sits_train

# sanity check: comprimento das séries antes de treinar
print(time_series["time_series"].apply(len).describe())

with mlflow.start_run(run_name="rfor-baseline"):
    mlflow.log_params(
        {
            "collection": "SENTINEL-2-16D",
            "bands": "NDVI",
            "start_date": START_DATE,
            "end_date": END_DATE,
            "n_samples": len(samples_df),
            "n_points_per_class": N_POINTS_PER_CLASS,
            "roi": str(ROI),
            "ml_method": "sits_rfor",
            "num_trees": 20,
            "kfolds": 5,
        }
    )

    kfold = sits_kfold_validate(
        samples=time_series,
        folds=5,
        ml_method=sits_rfor(num_trees=20),
        multicores=1,
    )
    print(type(kfold))
    print(kfold)

    model = sits_train(samples=time_series, ml_method=sits_rfor(num_trees=20))
    mlflow.log_param("trained", True)

count    120.0
mean       2.0
std        0.0
min        2.0
25%        2.0
50%        2.0
75%        2.0
max        2.0
Name: time_series, dtype: float64
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
  |======================================================================| 100%
<class 'pysits.models.data.matrix.SITSConfusionMatrix'>
Confusion Matrix and Statistics

                                                Reference
Prediction                                       VEGETACAO_NATURAL_PRIMARIA
  VEGETACAO_NATURAL_PRIMARIA                                              0
  OUTRAS_AREAS_EDIFICADAS                                                 0
  PASTAGEM                                                             

## 7. Classificação do cubo (probabilidades → mapa temático)

In [9]:
from pathlib import Path

OUTPUT_DIR = Path("../data/processed").resolve()
PROBS_DIR = OUTPUT_DIR / "probs"
BAYES_DIR = OUTPUT_DIR / "bayes"
CLASSIFIED_DIR = OUTPUT_DIR / "classified"

for directory in (PROBS_DIR, BAYES_DIR, CLASSIFIED_DIR):
    directory.mkdir(parents=True, exist_ok=True)
    print(directory, directory.exists(), directory.is_dir())

/home/sdesena/github/mlops_lulc/data/processed/probs True True
/home/sdesena/github/mlops_lulc/data/processed/bayes True True
/home/sdesena/github/mlops_lulc/data/processed/classified True True


In [ ]:
from pysits import sits_classify, sits_label_classification, sits_smooth

probs_cube = sits_classify(
    data=cube,
    ml_model=model,
    output_dir=str(PROBS_DIR),
    multicores=1,
    memsize=2,
    progress=True,
)

print(probs_cube)

  |                                                                      |   0%

In [ ]:
bayes_cube = sits_smooth(
    cube=probs_cube,
    output_dir=str(BAYES_DIR),
    multicores=1,
    progress=True,
)

print(bayes_cube)

In [ ]:
label_cube = sits_label_classification(
    cube=bayes_cube,
    output_dir=str(CLASSIFIED_DIR),
    progress=True,
)

print(label_cube)

## 8. Incerteza (entropia) — base para Active Learning (opcional)

In [ ]:
from pysits import sits_uncertainty

unc_cube = sits_uncertainty(cube=probs_cube, type_="entropy")
print(unc_cube)